# Part 3 — One YAML is the experiment

> **Extra, read only.** This lecture notebook is kept with its saved outputs. It ran with anemoi-training 0.17 and anemoi-inference 0.12 on the lecture dataset, not with the course kernel, so read it but do not rerun it in class.

**Slides 15–23 · 12 minutes.** Question: where in `course.yaml` is each term of the method, and what does `validate` catch when the file is wrong?
Runs on a login node: composition and validation touch neither a GPU nor the dataset.
Prerequisites: the course environment with `anemoi-training` 0.17.0 (runbook Phase 0) and `$ANEMOI_COURSE_BASE/config/course.yaml` (runbook Phase 3).
Tags: `# RUN` = execute now · `# REVIEW` = shown in the 30-minute review.

## 0 · Setup

Paths and one helper that calls the `anemoi-training` CLI from the config directory.

In [1]:
# RUN — paths and environment
import os, re, shutil, subprocess, sys
from pathlib import Path

from omegaconf import OmegaConf
from hydra import initialize_config_dir, compose
import anemoi.training

# where things are: <repo>/data/anemoi-course, or the folder named in ANEMOI_COURSE_BASE
BASE = Path(os.environ.get("ANEMOI_COURSE_BASE", "../../../data/anemoi-course")).resolve()
DATA = Path(os.environ.get("ANEMOI_COURSE_DATA", BASE / "data" / "era5-o48-2020-2021-6h-v1.zip"))   # the course dataset (Part 2)
GRAPH = Path(os.environ.get("ANEMOI_COURSE_GRAPH", BASE / "graph" / "graph_o48_tri4.pt"))          # the graph built in Part 2
CFG = BASE / "config"                                     # the generated Hydra tree lives here
CFG.mkdir(parents=True, exist_ok=True)                    # create the folder on a fresh machine
CLI = shutil.which("anemoi-training")                     # the command-line tool of this environment
if CLI is None:
    CLI = str(Path(sys.executable).parent / "anemoi-training")   # fall back to the venv's bin folder

if DATA.exists():
    dataset_status = "ok"
else:
    dataset_status = "missing"
if GRAPH.exists():
    graph_status = "ok"
else:
    graph_status = "missing"

print("anemoi-training", anemoi.training.__version__, "| config dir", CFG)
print("dataset", dataset_status, "| graph", graph_status, "(paths only; not opened here)")

anemoi-training 0.17.0 | config dir /gpfs/scratch/bsc32/bsc437208/anemoi-course/config
dataset ok | graph ok (paths only; not opened here)


**Expected:** `anemoi-training 0.17.0`, dataset `ok`, graph `ok`. Nothing is opened in this part.

In [2]:
# RUN — one helper that runs `anemoi-training <args>` and prints a short extract of its log
def cli(*args, tail=4, grep=None):
    # run inside CFG so the cwd search-path plugin finds our files; return the exit code
    result = subprocess.run([CLI, *args], cwd=CFG, capture_output=True, text=True)
    output = result.stdout + result.stderr                # the CLI logs to stderr, its errors too
    lines = []
    for line in output.strip().splitlines():
        if "further information" in line:
            continue                                      # a pydantic footer with a URL; skip it
        if grep is not None and not re.search(grep, line):
            continue                                      # with `grep`, keep only the lines that match
        lines.append(line)
    if len(lines) <= tail:
        shown = lines                                     # short log: print all of it
    else:
        shown = lines[: tail - 2] + ["  …"] + lines[-2:]  # long log: the first lines, "…", the last two
    print("\n".join(shown))
    return result.returncode

**Expected:** nothing printed; the helper is used from section 1 on.

## 1 · `config generate`: the eight groups (slides 17, 20)

The CLI copies the packaged tree into the current directory, one folder per group (existing files are skipped).
`task/forecaster.yaml` is the group file that defines one training sample in time.

In [3]:
# RUN — generate the tree (idempotent), list the groups, read the task file
cli("config", "generate", tail=1)                         # copies the packaged tree into CFG; existing files are skipped

groups = []
for path in CFG.iterdir():
    if path.is_dir() and path.name != "outputs":          # one folder per group; `outputs` is Hydra's own log folder
        groups.append(path.name)
groups = sorted(groups)
print("groups:", len(groups), groups, "\n")

task_file = CFG / "task" / "forecaster.yaml"              # the group file that defines one training sample
print(task_file.read_text())

2026-09-17 12:53:49 INFO Generating configs, please wait.
2026-09-17 12:53:49 INFO File /gpfs/scratch/bsc32/bsc437208/anemoi-course/config/multi.yaml already exists, skipping
2026-09-17 12:53:49 INFO File /gpfs/scratch/bsc32/bsc437208/anemoi-course/config/temporal_downscaler.yaml already exists, skipping
2026-09-17 12:53:49 INFO File /gpfs/scratch/bsc32/bsc437208/anemoi-course/config/transport_stochastic_interpolant.yaml already exists, skipping
2026-09-17 12:53:49 INFO File /gpfs/scratch/bsc32/bsc437208/anemoi-course/config/config.yaml already exists, skipping
2026-09-17 12:53:49 INFO File /gpfs/scratch/bsc32/bsc437208/anemoi-course/config/hierarchical_autoencoder.yaml already exists, skipping
2026-09-17 12:53:49 INFO File /gpfs/scratch/bsc32/bsc437208/anemoi-course/config/hierarchical.yaml already exists, skipping
2026-09-17 12:53:49 INFO File /gpfs/scratch/bsc32/bsc437208/anemoi-course/config/multi_transport_edm_diffusion.yaml already exists, skipping
2026-09-17 12:53:49 INFO File /

**Expected:** 8 groups — `data dataloader diagnostics graph model system task training`. The sample: `multistep_input: 2`
past states in, `multistep_output: 1` state out, `timestep: "6H"`, `rollout.maximum: 1` repetition.

## 2 · The course configuration: nine `defaults` lines plus the differences (slide 23)

`course.yaml` picks one file per group (`graphtransformer` instead of `gnn`) and lists only what differs from them.
`tracking_uri` must not stay `???`: validation resolves every value, even with MLflow disabled (slide 22).

In [4]:
# RUN — the file the whole course trains from
COURSE_PATH = CFG / "course.yaml"                         # written by runbook Phase 3
assert COURSE_PATH.exists(), "run runbook Phase 3 first (it writes course.yaml)"
course_text = COURSE_PATH.read_text()                     # kept as a string: section 5 derives two broken copies from it
n_course = len(course_text.splitlines())
print(course_text)
print(f"── {n_course} lines")

defaults:
- data: zarr
- dataloader: native_grid
- diagnostics: evaluation
- system: example
- graph: multi_scale
- model: graphtransformer
- task: forecaster
- training: single
- _self_

config_validation: True

system:
  input:
    dataset: /gpfs/scratch/bsc32/bsc437208/anemoi-course/data/era5-o48-2020-2021-6h-v1.zarr
    graph: /gpfs/scratch/bsc32/bsc437208/anemoi-course/graph/graph_o48_tri4.pt
  output:
    root: /gpfs/scratch/bsc32/bsc437208/anemoi-course/runs
  hardware:
    accelerator: cuda
    num_gpus_per_node: 1
    num_nodes: 1
    num_gpus_per_model: 1

graph:
  nodes:
    hidden:
      node_builder:
        resolution: 4

dataloader:
  batch_size: {training: 2, validation: 4}
  num_workers: {training: 8, validation: 8, test: 8}
  limit_batches: {training: 100, validation: 50}
  training:   {datasets: {data: {start: null, end: 2020}}}
  validation: {datasets: {data: {start: 2021-01, end: 2021-06}}}
  test:       {datasets: {data: {start: 2021-07, end: null}}}

model:
  num

**Expected:** 55 lines (slide 23's shorter version: 30). Five blocks of differences; `system` (paths, one GPU) is the
only machine-specific one, everything else would be the same on any machine.

## 3 · `config dump`: the resolved tree (slides 16, 22)

`dump` composes the tree exactly as `train` would and writes it out. Read this file whenever a run surprises you.

In [5]:
# REVIEW — compose without training
RESOLVED = CFG / "resolved.yaml"                          # where `dump` writes the composed tree
dump_args = ["config", "dump", "--config-path", str(CFG), "--config-name", "course", "--output", str(RESOLVED), "--overwrite"]
cli(*dump_args, tail=1)                                   # same as `anemoi-training config dump ...` in a terminal
n_resolved = len(RESOLVED.read_text().splitlines())
print(f"{RESOLVED.name}: {n_resolved} lines")

2026-09-17 12:53:51 INFO Dumping config to /gpfs/scratch/bsc32/bsc437208/anemoi-course/config/resolved.yaml
2026-09-17 12:53:51 INFO Appending current working directory (/gpfs/scratch/bsc32/bsc437208/anemoi-course/config) to the search path.
2026-09-17 12:53:51 INFO Search path is now: [provider=hydra, path=pkg://hydra.conf, provider=main, path=/gpfs/scratch/bsc32/bsc437208/anemoi-course/config, provider=anemoi-cwd-searchpath-plugin, path=/gpfs/scratch/bsc32/bsc437208/anemoi-course/config, provider=anemoi-package-searchpath-plugin, path=pkg://anemoi.training/config]
  …
2026-09-17 12:53:51 INFO Search path is now: [provider=hydra, path=pkg://hydra.conf, provider=main, path=/gpfs/scratch/bsc32/bsc437208/anemoi-course/config, provider=anemoi-cwd-searchpath-plugin, path=/gpfs/scratch/bsc32/bsc437208/anemoi-course/config, provider=anemoi-package-searchpath-plugin, path=pkg://anemoi.training/config]
2026-09-17 12:53:51 INFO Dumping file in /gpfs/scratch/bsc32/bsc437208/anemoi-course/config/

**Expected:** 588 lines (slide 16 quotes 533 for the 30-line `course.yaml`). Nine `defaults` lines became the
complete parameter list; `train` composes the same tree at start-up.

## 4 · Central exercise A: one key per term (slide 15)

Eight terms: dataset · timestep, inputs, rollout · hidden resolution · width, depth, processor · normalizer · loss, scalers · optimization · hardware, output.
Write the key for each on paper first, then run.

In [6]:
# RUN — the eight rows of slide 15: key → value in resolved.yaml
cfg = OmegaConf.load(RESOLVED)                            # the resolved tree, as an OmegaConf object
raw = OmegaConf.to_container(cfg, resolve=False)          # plain dict; `${...}` references left as written
res = OmegaConf.to_container(cfg, resolve=True)           # plain dict; `${...}` references replaced by their values


def get(tree, key):
    # walk a dotted key such as "model.num_channels" down the nested dict
    node = tree
    for part in key.split("."):
        node = node[part]
    return node


ROWS = [
    ("dataset",                   ["system.input.dataset"]),
    ("timestep, inputs, rollout", ["task.timestep", "task.multistep_input", "task.rollout.maximum"]),
    ("hidden resolution",         ["graph.nodes.hidden.node_builder.resolution"]),
    ("width, depth, processor",   ["model.num_channels", "model.processor.num_layers", "model.processor._target_"]),
    ("normalizer",                ["data.datasets.data.processors.normalizer.config.default"]),
    ("loss, scalers",             ["training.training_loss.datasets.data._target_", "training.training_loss.datasets.data.scalers"]),
    ("optimization",              ["training.optimization.optimizer._target_", "training.optimization.lr", "training.optimization.lr_scheduler.t_initial"]),
    ("hardware, output",          ["system.hardware.num_gpus_per_node", "system.output.root", "diagnostics.checkpoint.every_n_epochs.save_frequency"]),
]
for term, keys in ROWS:
    for key in keys:
        written = get(raw, key)                           # the value as written in the YAML files
        resolved = get(res, key)                          # the value after `${...}` resolution
        value = str(resolved)[:40]                        # long paths are cut at 40 characters
        note = ""
        if written != resolved:
            note = f"   ← written as {written}"           # only the interpolated keys get this note
        print(f"{term:26s} {key:58s} {value}{note}")

dataset                    system.input.dataset                                       /gpfs/scratch/bsc32/bsc437208/anemoi-cou
timestep, inputs, rollout  task.timestep                                              6H
timestep, inputs, rollout  task.multistep_input                                       2
timestep, inputs, rollout  task.rollout.maximum                                       1
hidden resolution          graph.nodes.hidden.node_builder.resolution                 4
width, depth, processor    model.num_channels                                         128
width, depth, processor    model.processor.num_layers                                 16
width, depth, processor    model.processor._target_                                   anemoi.models.layers.processor.GraphTran
normalizer                 data.datasets.data.processors.normalizer.config.default    mean-std
loss, scalers              training.training_loss.datasets.data._target_              anemoi.training.losses.MSELoss
lo

**Expected:** `6H`, `2`, `1`; resolution `4`; `128`, `16`, `GraphTransformerProcessor`; `mean-std`; `MSELoss` with three scalers;
`AdamW`, `lr 6.25e-05`, `t_initial` written as `${training.max_steps}` = `150000`; 1 GPU, one checkpoint per epoch.
The schedule spans 150,000 steps although `max_epochs: 5` × `limit_batches 100` stops the run at step 500, inside the 1,000-step warm-up.

> **Checkpoint:** which key would you change so that the schedule ends when the run ends?

## 5 · Central exercise B: break it two ways, then check what `validate` did not (slide 22)

`course_bad`: `num_channels: many` and an unknown key `max_stepz`. `course_silent`: `soil_moisture` added to the forcing list.
Predict each verdict before running: accepted or rejected, on which key, with which message?

In [7]:
# RUN — write two broken copies of course.yaml
# 1) course_bad: a wrong type and an unknown key
bad_text = course_text.replace("num_channels: 128", "num_channels: many")          # an integer becomes a word
bad_text = bad_text.replace("max_epochs: 5", "max_epochs: 5\n  max_stepz: 10")    # a misspelt key under `training:`
BAD_PATH = CFG / "course_bad.yaml"
BAD_PATH.write_text(bad_text)

# 2) course_silent: a variable the dataset does not have, added to the forcing list
forcing = list(cfg.data.datasets.data.forcing)            # the forcing names of the resolved config
forcing.append("soil_moisture")                           # one more name, not in the dataset
silent_block = f"\ndata:\n  datasets:\n    data:\n      forcing: {forcing}\n"     # one more block, merged last
silent_text = course_text + silent_block
SILENT_PATH = CFG / "course_silent.yaml"
SILENT_PATH.write_text(silent_text)

print(BAD_PATH.name, "- the two changed lines:")
for line in bad_text.splitlines():
    if "num_channels" in line or "max_stepz" in line:
        print("   ", line)
print(SILENT_PATH.name, "- forcing names:", len(forcing), "(the last one is", forcing[-1] + ")")

course_bad.yaml - the two changed lines:
      num_channels: many
      max_stepz: 10
course_silent.yaml - forcing names: 14 (the last one is soil_moisture)


**Expected:** `course_bad.yaml` shows `num_channels: many` and `max_stepz: 10`; `course_silent.yaml` ends with a forcing list whose last name is `soil_moisture`.

In [8]:
# RUN — three verdicts from `validate`, then one look into the dataset
from anemoi.datasets import open_dataset

# the log lines worth reading: the verdict, the error count, the two broken keys and their error types
PATTERN = r"validated\.|validation errors|Processor`\.num_channels$|max_stepz$|int_parsing|extra_forbidden"
for name in ("course", "course_bad", "course_silent"):
    print(f"══ {name}")
    cli("config", "validate", f"--config-name={name}", tail=6, grep=PATTERN)

dataset = open_dataset(DATA)                              # reads the metadata only; no field is loaded
variables = dataset.variables                             # the names in the dataset
has_soil_moisture = "soil_moisture" in variables
print("\nvariables in the dataset:", len(variables), "| 'soil_moisture' in dataset:", has_soil_moisture)

══ course


2026-09-17 12:53:54 INFO Config files validated.
══ course_bad


pydantic_core._pydantic_core.ValidationError: 19 validation errors for BaseSchema
model.BaseModelSchema.processor.`anemoi.models.layers.processor.GraphTransformerProcessor`.num_channels
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='many', input_type=str]
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='many', input_type=str]
  …
training.`anemoi.training.train.methods.SingleTraining`.max_stepz
  Extra inputs are not permitted [type=extra_forbidden, input_value=10, input_type=int]
══ course_silent


2026-09-17 12:54:00 INFO Config files validated.



variables in the dataset: 43 | 'soil_moisture' in dataset: False


**Expected:** `course` and `course_silent` print `Config files validated.`; `course_bad` reports `19 validation errors`: `int_parsing` on
`num_channels` (one value interpolated into processor, encoder and decoder, repeated per model-schema variant) and `extra_forbidden` on `max_stepz`.
The dataset has 43 variables and no `soil_moisture`: `validate` checks types and keys, never the data, so `course_silent` dies in the first minute of training.

## 6 · One override without running (slides 16, 22)

Everything after `train` goes to Hydra: `group=file` swaps a whole group file, `key=value` changes one leaf.
The cell composes `course.yaml` with `model=transformer` in Python, exactly as `train` would.

In [9]:
# RUN — swap the model group; the leaf set in course.yaml still wins
with initialize_config_dir(config_dir=str(CFG), version_base=None):             # Hydra searches our config folder
    composed = compose(config_name="course", overrides=["model=transformer"])    # same as `train model=transformer`
print("processor   :", composed.model.processor._target_)
print("num_channels:", composed.model.num_channels)

processor   :

 anemoi.models.layers.processor.TransformerProcessor
num_channels: 128


**Expected:** `TransformerProcessor` (from `model/transformer.yaml`) and `num_channels` still `128`: `course.yaml` is merged last (`_self_`),
so its leaves win over any group file. One line, one controlled experiment.

> **Checkpoint:** batch size halved from 2 to 1 — which key keeps the experiment comparable (slide 20)?

## 7 · What `validate` does not check

A green `Config files validated.` proves the tree is complete and well-typed. It does not prove that

- the dataset is readable from the compute node and contains every name in `forcing` / `diagnostic` (`course_silent`),
- the model fits in GPU memory at this `batch_size`, `num_channels`, `resolution`,
- `system.input.graph` was built from the same grid and dataset as `system.input.dataset`.

Part 4 opens with a pre-launch cell that checks the first one before `sbatch`.